In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import warnings
import matplotlib
import matplotlib.pyplot as plt
import sys 
import os 
import pandas as pd
import anndata as ad
import scanpy as sc
import numpy as np
import seaborn as sns
from tqdm import tqdm
import pandas as pd
from scipy.stats import linregress
import matplotlib.patches as mpatches
from scipy.stats import pearsonr, spearmanr
from pandas.api.types import CategoricalDtype
from scipy.cluster.hierarchy import linkage
from matplotlib.patches import Patch
from statsmodels.stats.multitest import multipletests



plt.rcParams["figure.figsize"]=4,4

warnings.filterwarnings("ignore")



# Extend collecTRI by adding indivitual sources to edges

In [8]:
from ciim.src.utils.util import flesh_out_collectri
flesh_out_collectri()

In [9]:
curated_net = pd.read_csv('/vol/projects/jnourisa/prior/collectri_with_source.csv')
curated_net['ref'].unique()

array(['DoRothEA-A', 'ExTRI', 'HTRI', 'Pavlidis2021', 'TFactS', 'TRRUST',
       'GEREDB', 'CytReg', 'SIGNOR', 'GOA', 'IntAct'], dtype=object)

# Valid gene names

In [ ]:
def get_known_gene_names():
    import pandas as pd
    gtf_cols = [
        'seqname', 'source', 'feature', 'start', 'end',
        'score', 'strand', 'frame', 'attribute'
    ]

    gtf_path = '/home/jnourisa/projs/ongoing/task_grn_inference/resources/supp_data/gencode.v47.annotation.gtf.gz'

    gtf_df = pd.read_csv(
        gtf_path,
        sep='\t',
        compression='gzip',
        comment='#',
        header=None,
        names=gtf_cols,
        engine='python'
    )
    import re

    # Define known chromosomes (as strings)
    known_chromosomes = [f'chr{i}' for i in range(1, 21)] + ['X', 'Y']

    # Filter rows where feature is 'gene' and on a known chromosome
    genes_df = df_only_genes[
        (gtf_df['feature'] == 'gene') &
        (gtf_df['seqname'].isin(known_chromosomes))
    ].copy()

    # Function to extract gene_name from the attribute column
    def extract_gene_name(attr_str):
        match = re.search(r'gene_name "([^"]+)"', attr_str)
        return match.group(1) if match else None

    # Apply the function
    genes_df['gene_name'] = genes_df['attribute'].apply(extract_gene_name)

    # Drop NA (just in case) and get unique gene names
    gene_names = genes_df['gene_name'].dropna().unique().tolist()
    np.savetxt(
        '/vol/projects/jnourisa/prior/gene_names.txt',
        gene_names,
        fmt='%s'
    )
get_known_gene_names()

# Promotor based skeleton

In [2]:
net1 = pd.read_csv(f'/home/jnourisa/projs/ongoing/task_grn_inference/output/skeleton/skeleton_encode_promotor.csv')
net1['edge'] = net1['source'] + '_' + net1['target']

net2 = pd.read_csv(f'/home/jnourisa/projs/ongoing/task_grn_inference/output/skeleton/skeleton_jaspar_promotor.csv')
net2['edge'] = net2['source'] + '_' + net2['target']



In [9]:
net = pd.concat([net1, net2])
net.drop_duplicates(subset=['edge'], inplace=True)
net = net[['source', 'target', 'edge']].reset_index(drop=True)

In [11]:
net.to_csv(f'/vol/projects/jnourisa/prior/skeleton_promotor.csv', index=False)